In [ ]:
import sys
import os
import numpy as np

sys.path.append(os.path.abspath("../src"))
from walinet.visualization.MetabMapsVis import * 

In [ ]:
B0 = np.load("../B0_correction/Vol5/B0_estimation.npy")[...,0]

B0 = np.swapaxes(B0,0,1)

B0.shape

# 3T

In [ ]:
mat_paths = [
    "MetabMapsProton/3T/Maps/50x50/Vol1_b4Wali.mat",
    #"MetabMapsProton/3T/Maps/50x50/Vol1_Wali_1.1_5Layer.mat",
    "MetabMapsProton/3T/Maps/50x50/Vol1_Ynet_1.1.mat",
    #"MetabMapsProton/3T/Maps/50x50/Vol1_Unet_1.1.mat",
    #"MetabMapsProton/3T/Maps/50x50/Vol1_Wali_1.1_6Layer.mat",
    ]

col_titles = [
    "3T Vol01 Before Walinet",
    "3T Vol01 Ynet new",
    #"3T Vol01 Unet new",
    #"3T Vol01 Unet old",
    ]

mask = np.load("/workspace/Denoising/datasets/Proton/3T/Vol01_WB/Res50x50/masks/brain_mask.npy")

Ref = np.load("/workspace/Denoising/datasets/Proton/3T/Vol01_WB/Res50x50/OriginalData/magnitude.npy")

Ref =  np.transpose(Ref, (2, 0, 1))

slice_z = 14

plot_lcmodel_comparison(mat_paths, col_titles, slice_z=slice_z, save_path="SavedGraphics/lcmodel_comparison.pdf", ref_image = Ref, ref_title = "Magnitude")

In [ ]:
Ref.shape

# 7T

In [ ]:
mat_paths = [
    "MetabMapsProton/7T/Maps/MS_180_B4_Walinet.mat",
    "MetabMapsProton/7T/Maps/MS_180_After_Walinet.mat",  
    "MetabMapsProton/7T/Maps/MS_180_WALI_558_Unet6Layer_3.mat",
    "MetabMapsProton/7T/Maps/MS_180_Ynet_6Layer.mat"
    #"MetabMapsProton/7T/Maps/MS_180_WALI_558.mat",
    #"MetabMapsProton/7T/Maps/MS_180_Wali_Unet_6Layer.mat",   
    #"MetabMapsProton/7T/Maps/AllMaps_B0_correction_after_Walinet.mat",
    #"MetabMapsProton/7T/Maps/5_B0_corrected_walinet_abs_max.mat",
    #"MetabMapsProton/7T/Maps/5_Wali_7Layers_old.mat"
    ]

col_titles = [
    "MS 180 No Walinet No L2",
    "MS 180 Scanner Walinet",
    "MS 180 Unet 558 deep 3",
    "MS 180 Ynet 558 deep"
    #"MS 180 New Walinet 558",
    #"MS 180 New Walinet",
    #"7T Vol5 Denoised B4 Walinet",
    #"7T Vol5 After Walinet",
    #"7T Vol 5 Unet abs max",
    #"7 Layers"
    ]

mask = np.load("/workspace/Denoising/datasets/Proton/7T/NoB0Correction/MS_180/masks/brain_mask.npy")

Ref = np.load("/workspace/Denoising/datasets/Proton/7T/NoB0Correction/MS_180/OriginalData/magnitude.npy")

Ref =  np.swapaxes(np.transpose(Ref, (2, 0, 1)),1,2)

# mask = np.load("/workspace/Denoising/datasets/Proton/7T/B0corrected_wo_LipidMask/Vol5/masks/brain_mask.npy")

# Ref = np.load("/workspace/Denoising/datasets/Proton/7T/B0corrected_wo_LipidMask/Vol5/OriginalData/magnitude.npy")

# Ref =  np.transpose(Ref, (2, 0, 1))

slice_z = 16

plot_lcmodel_comparison(mat_paths, col_titles, slice_z=slice_z, save_path="SavedGraphics/lcmodel_comparison.pdf", ref_image = Ref, ref_title = "Magnitude")

In [ ]:
import nibabel as nib

img = nib.load("Orig/NAA_amp_map.nii")
data = img.get_fdata()

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

metabo = "Glu"

nii_paths = [
    f"Orig_L2/{metabo}_amp_map.nii",
    f"Orig/{metabo}_amp_map.nii.gz",
]

titles = [
    "L2",
    "WALINET",
]

slice_z = 17
low_pct = 0
high_pct = 99

fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for ax, nii_path, title in zip(axes, nii_paths, titles):
    img = nib.load(nii_path)
    data = img.get_fdata()

    slice_2d = data[:, :, slice_z]

    valid = slice_2d[np.isfinite(slice_2d)]
    vmin, vmax = np.percentile(valid, (low_pct, high_pct))

    im = ax.imshow(
        slice_2d.T,
        cmap="plasma",
        origin="lower",
        vmin=vmin,
        vmax=vmax,
    )

    ax.set_title(f"{title} — z={slice_z}")
    ax.axis("off")

    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

B0 = B0*mask

n_slices = B0.shape[2]
n_cols = 5
n_rows = math.ceil(n_slices / n_cols)

vmin, vmax = np.nanpercentile(B0, [0, 100])

fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 3*n_rows))
axes = axes.ravel()

for z in range(n_slices):
    im = axes[z].imshow(
        B0[:, :, z].T,
        origin="lower",
        cmap="turbo",   # oder "viridis", "magma", "seismic"
        vmin=vmin,
        vmax=vmax,
    )
    axes[z].set_title(f"z={z}")
    axes[z].axis("off")

for ax in axes[n_slices:]:
    ax.axis("off")

fig.colorbar(im, ax=axes[:n_slices], fraction=0.02, pad=0.01, label="B0 inhomogeneity")
plt.tight_layout()
plt.show()